# LM Evaluation Harness를 활용한 파인튜닝 모델 평가

## 목차
1. [lm-evaluation-harness 소개](#1-lm-evaluation-harness-소개)
2. [환경 설정 및 설치](#2-환경-설정-및-설치)
3. [Hugging Face 모델 로드](#3-hugging-face-모델-로드)
4. [커스텀 평가 데이터셋 준비](#4-커스텀-평가-데이터셋-준비)
5. [평가 태스크 설정](#5-평가-태스크-설정)
6. [평가 실행](#6-평가-실행)
7. [결과 분석 및 해석](#7-결과-분석-및-해석)

---

## 1. lm-evaluation-harness 소개

**lm-evaluation-harness**는 EleutherAI에서 개발한 오픈소스 LLM 평가 프레임워크입니다.

### 주요 특징
- ✅ **표준화된 평가**: 200개 이상의 표준 벤치마크 태스크 지원
- ✅ **객관적인 비교**: 다른 모델과 공정한 성능 비교 가능
- ✅ **자동 채점**: MCQ(Multiple Choice Questions) 중심의 자동 평가
- ✅ **커스텀 태스크**: 사용자 정의 평가 데이터셋 추가 가능
- ✅ **재현성**: 평가 결과의 재현성 보장

### 평가 대상 모델
- **모델명**: `good593/EXAONE-3.5-2.4B-fine-tuning`
- **베이스 모델**: EXAONE-3.5-2.4B
- **특화 도메인**: 한국어 의료/건강 상담

### 평가 데이터셋
- `illnesses.csv`: 질병 관련 Q&A (795개 샘플)
- `nutrition.csv`: 영양/건강 관련 Q&A (200개 샘플)


## 2. 환경 설정 및 설치

lm-evaluation-harness를 사용하기 위한 환경을 설정합니다.


In [ ]:
# lm-evaluation-harness 설치
%pip install lm-eval==0.4.3 -q

# 필요한 라이브러리 설치
%pip install transformers torch pandas accelerate -q

print("✅ 설치 완료!")


In [ ]:
# 필요한 라이브러리 임포트
import os
import json
import pandas as pd
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# GPU 사용 확인
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 사용 중인 디바이스: {device}")

# 작업 디렉토리 설정
workspace_dir = Path.cwd()
data_dir = workspace_dir / "data"
print(f"📁 작업 디렉토리: {workspace_dir}")
print(f"📊 데이터 디렉토리: {data_dir}")


## 3. Hugging Face 모델 로드

파인튜닝된 모델을 Hugging Face Hub에서 로드합니다.


In [ ]:
# 모델 정보
MODEL_NAME = "good593/EXAONE-3.5-2.4B-fine-tuning"

print(f"📥 모델 로드 중: {MODEL_NAME}")

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True
)

print(f"✅ 모델 로드 완료!")
print(f"📊 모델 파라미터 수: {model.num_parameters():,}")

# 간단한 테스트
test_input = "소아청소년과 전문의로서, 열이 나는 아이를 어떻게 돌봐야 하나요?"
inputs = tokenizer(test_input, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=100)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"\n🧪 테스트 입력: {test_input}")
print(f"💬 모델 응답: {response[:200]}...")


## 4. 커스텀 평가 데이터셋 준비

질병 및 영양 관련 데이터셋을 lm-evaluation-harness 형식으로 변환합니다.


In [ ]:
# 데이터셋 로드
illnesses_df = pd.read_csv(data_dir / "illnesses.csv")
nutrition_df = pd.read_csv(data_dir / "nutrition.csv")

print(f"📊 질병 데이터셋: {len(illnesses_df)}개 샘플")
print(f"📊 영양 데이터셋: {len(nutrition_df)}개 샘플")
print(f"📊 전체: {len(illnesses_df) + len(nutrition_df)}개 샘플")

# 데이터 구조 확인
print("\n🔍 데이터셋 컬럼:")
print(illnesses_df.columns.tolist())

# 샘플 데이터 확인
print("\n📄 샘플 데이터 (질병):")
sample = illnesses_df.iloc[0]
print(f"질문: {sample['user_input'][:100]}...")
print(f"정답: {sample['reference'][:100]}...")

print("\n📄 샘플 데이터 (영양):")
sample = nutrition_df.iloc[0]
print(f"질문: {sample['user_input'][:100]}...")
print(f"정답: {sample['reference'][:100]}...")


In [ ]:
# lm-eval 형식으로 데이터셋 변환
def prepare_eval_dataset(df, task_name):
    """
    DataFrame을 lm-eval 형식의 JSONL로 변환
    """
    eval_data = []
    
    for idx, row in df.iterrows():
        eval_data.append({
            "doc_id": idx,
            "query": row["user_input"],
            "reference": row["reference"],
            "persona": row.get("persona_name", ""),
            "query_style": row.get("query_style", ""),
            "query_length": row.get("query_length", "")
        })
    
    return eval_data

# 데이터셋 변환
illnesses_eval = prepare_eval_dataset(illnesses_df, "medical_illnesses")
nutrition_eval = prepare_eval_dataset(nutrition_df, "medical_nutrition")

print(f"✅ 질병 평가 데이터: {len(illnesses_eval)}개")
print(f"✅ 영양 평가 데이터: {len(nutrition_eval)}개")

# 샘플 확인
print("\n📄 변환된 샘플:")
print(json.dumps(illnesses_eval[0], ensure_ascii=False, indent=2))


## 5. 평가 태스크 설정

lm-evaluation-harness를 사용하여 커스텀 평가 태스크를 설정합니다.

### 평가 방식

이 강의에서는 **생성형 평가(Generative Evaluation)**를 사용합니다:

1. **모델 응답 생성**: 모델이 질문에 대한 자유로운 형태의 답변을 생성
2. **품질 평가**: 생성된 답변을 정답(reference)과 비교하여 평가

### 평가 지표

- **ROUGE**: 생성된 답변과 정답의 단어/구문 중복도
  - ROUGE-1: 단어 단위 중복
  - ROUGE-2: 2-gram 중복
  - ROUGE-L: 최장 공통 부분 수열
- **BLEU**: 기계 번역 품질 측정 (n-gram 기반)
- **정성 평가**: 실제 답변 내용 확인


In [ ]:
# 평가 메트릭 설치
%pip install rouge-score nltk -q

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

from rouge_score import rouge_scorer

print("✅ 평가 메트릭 준비 완료!")


## 6. 평가 실행

실제 평가를 수행하고 결과를 수집합니다.


In [ ]:
def generate_response(model, tokenizer, query, max_new_tokens=256):
    """
    모델을 사용하여 질문에 대한 응답 생성
    """
    inputs = tokenizer(query, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 질문 부분 제거
    response = response.replace(query, "").strip()
    return response

def evaluate_dataset(eval_data, dataset_name, num_samples=None):
    """
    데이터셋 평가 실행
    """
    if num_samples:
        eval_data = eval_data[:num_samples]
    
    results = []
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    print(f"\n🔍 {dataset_name} 평가 시작 ({len(eval_data)}개 샘플)...")
    
    for idx, item in enumerate(eval_data):
        if (idx + 1) % 10 == 0:
            print(f"진행 중: {idx + 1}/{len(eval_data)}")
        
        query = item["query"]
        reference = item["reference"]
        
        # 모델 응답 생성
        response = generate_response(model, tokenizer, query)
        
        # ROUGE 점수 계산
        scores = scorer.score(reference, response)
        
        results.append({
            "doc_id": item["doc_id"],
            "query": query,
            "reference": reference,
            "response": response,
            "rouge1_f": scores['rouge1'].fmeasure,
            "rouge2_f": scores['rouge2'].fmeasure,
            "rougeL_f": scores['rougeL'].fmeasure
        })
    
    print(f"✅ {dataset_name} 평가 완료!")
    return results

# 참고: 전체 데이터셋 평가는 시간이 오래 걸리므로 샘플 수를 제한합니다
# 전체 평가를 원하면 num_samples=None으로 설정하세요

print("⏳ 평가 시작... (샘플 수를 제한하여 빠른 실행)")
illnesses_results = evaluate_dataset(illnesses_eval, "질병 데이터셋", num_samples=20)
nutrition_results = evaluate_dataset(nutrition_eval, "영양 데이터셋", num_samples=20)


## 7. 결과 분석 및 해석

평가 결과를 분석하고 모델 성능을 해석합니다.


In [ ]:
### 7.1 전체 평균 점수

import numpy as np

def calculate_metrics(results):
    """
    평가 결과의 평균 점수 계산
    """
    rouge1_scores = [r['rouge1_f'] for r in results]
    rouge2_scores = [r['rouge2_f'] for r in results]
    rougeL_scores = [r['rougeL_f'] for r in results]
    
    return {
        "ROUGE-1 F1": np.mean(rouge1_scores),
        "ROUGE-2 F1": np.mean(rouge2_scores),
        "ROUGE-L F1": np.mean(rougeL_scores)
    }

# 데이터셋별 평균 점수
print("=" * 60)
print("📊 질병 데이터셋 평균 점수")
print("=" * 60)
illnesses_metrics = calculate_metrics(illnesses_results)
for metric, score in illnesses_metrics.items():
    print(f"  {metric}: {score:.4f}")

print("\n" + "=" * 60)
print("📊 영양 데이터셋 평균 점수")
print("=" * 60)
nutrition_metrics = calculate_metrics(nutrition_results)
for metric, score in nutrition_metrics.items():
    print(f"  {metric}: {score:.4f}")

# 전체 평균
all_results = illnesses_results + nutrition_results
print("\n" + "=" * 60)
print("📊 전체 평균 점수")
print("=" * 60)
overall_metrics = calculate_metrics(all_results)
for metric, score in overall_metrics.items():
    print(f"  {metric}: {score:.4f}")


In [ ]:
### 7.2 샘플 응답 확인

# 질병 데이터셋 샘플
print("=" * 80)
print("🩺 질병 데이터셋 샘플 응답")
print("=" * 80)
for i in range(3):
    result = illnesses_results[i]
    print(f"\n[샘플 {i+1}]")
    print(f"질문: {result['query']}")
    print(f"\n정답: {result['reference'][:150]}...")
    print(f"\n모델 응답: {result['response'][:150]}...")
    print(f"\nROUGE-1: {result['rouge1_f']:.4f} | ROUGE-2: {result['rouge2_f']:.4f} | ROUGE-L: {result['rougeL_f']:.4f}")
    print("-" * 80)

# 영양 데이터셋 샘플
print("\n" + "=" * 80)
print("🥗 영양 데이터셋 샘플 응답")
print("=" * 80)
for i in range(3):
    result = nutrition_results[i]
    print(f"\n[샘플 {i+1}]")
    print(f"질문: {result['query']}")
    print(f"\n정답: {result['reference'][:150]}...")
    print(f"\n모델 응답: {result['response'][:150]}...")
    print(f"\nROUGE-1: {result['rouge1_f']:.4f} | ROUGE-2: {result['rouge2_f']:.4f} | ROUGE-L: {result['rougeL_f']:.4f}")
    print("-" * 80)


In [ ]:
### 7.3 성능 분포 시각화

%pip install matplotlib -q

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ROUGE 점수 분포 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['rouge1_f', 'rouge2_f', 'rougeL_f']
titles = ['ROUGE-1 F1', 'ROUGE-2 F1', 'ROUGE-L F1']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    illnesses_scores = [r[metric] for r in illnesses_results]
    nutrition_scores = [r[metric] for r in nutrition_results]
    
    axes[idx].hist(illnesses_scores, alpha=0.5, label='질병', bins=10, color='blue')
    axes[idx].hist(nutrition_scores, alpha=0.5, label='영양', bins=10, color='green')
    axes[idx].set_xlabel('점수')
    axes[idx].set_ylabel('빈도')
    axes[idx].set_title(title)
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 데이터셋별 평균 점수 비교
fig, ax = plt.subplots(figsize=(10, 6))

datasets = ['질병', '영양', '전체']
rouge1_scores = [illnesses_metrics['ROUGE-1 F1'], nutrition_metrics['ROUGE-1 F1'], overall_metrics['ROUGE-1 F1']]
rouge2_scores = [illnesses_metrics['ROUGE-2 F1'], nutrition_metrics['ROUGE-2 F1'], overall_metrics['ROUGE-2 F1']]
rougeL_scores = [illnesses_metrics['ROUGE-L F1'], nutrition_metrics['ROUGE-L F1'], overall_metrics['ROUGE-L F1']]

x = np.arange(len(datasets))
width = 0.25

ax.bar(x - width, rouge1_scores, width, label='ROUGE-1', color='skyblue')
ax.bar(x, rouge2_scores, width, label='ROUGE-2', color='lightcoral')
ax.bar(x + width, rougeL_scores, width, label='ROUGE-L', color='lightgreen')

ax.set_xlabel('데이터셋')
ax.set_ylabel('F1 점수')
ax.set_title('데이터셋별 ROUGE 점수 비교')
ax.set_xticks(x)
ax.set_xticklabels(datasets)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
### 7.4 결과 해석 및 개선 방향

print("=" * 80)
print("📈 결과 해석")
print("=" * 80)

print("""
### ROUGE 점수 해석 가이드

#### 점수 범위별 의미
- **0.0 ~ 0.2**: 매우 낮음 - 모델 응답이 정답과 거의 유사하지 않음
- **0.2 ~ 0.4**: 낮음 - 일부 내용이 일치하나 개선 필요
- **0.4 ~ 0.6**: 중간 - 합리적인 수준의 답변 품질
- **0.6 ~ 0.8**: 높음 - 정답과 상당히 유사한 답변
- **0.8 ~ 1.0**: 매우 높음 - 정답과 거의 동일한 답변

#### 평가 지표별 의미
- **ROUGE-1**: 단어 수준의 중복도 (전체적인 내용 유사성)
- **ROUGE-2**: 2-gram 중복도 (문맥 고려한 유사성)
- **ROUGE-L**: 최장 공통 부분수열 (문장 구조 유사성)

### 현재 모델 성능 분석
""")

# 성능 분석
avg_rouge1 = overall_metrics['ROUGE-1 F1']
avg_rouge2 = overall_metrics['ROUGE-2 F1']
avg_rougeL = overall_metrics['ROUGE-L F1']

if avg_rouge1 >= 0.6:
    print("✅ ROUGE-1 점수가 높음: 모델이 정답과 유사한 단어들을 잘 사용하고 있습니다.")
elif avg_rouge1 >= 0.4:
    print("⚠️ ROUGE-1 점수가 중간: 답변 품질은 합리적이지만 개선 여지가 있습니다.")
else:
    print("❌ ROUGE-1 점수가 낮음: 모델이 정답과 다른 단어를 주로 사용하고 있습니다.")

if avg_rouge2 >= 0.4:
    print("✅ ROUGE-2 점수가 양호: 모델이 문맥을 고려한 답변을 생성하고 있습니다.")
elif avg_rouge2 >= 0.2:
    print("⚠️ ROUGE-2 점수가 중간: 문맥 파악 능력을 개선할 필요가 있습니다.")
else:
    print("❌ ROUGE-2 점수가 낮음: 모델의 문맥 이해 능력이 부족합니다.")

if avg_rougeL >= 0.5:
    print("✅ ROUGE-L 점수가 높음: 모델이 문장 구조를 잘 이해하고 있습니다.")
elif avg_rougeL >= 0.3:
    print("⚠️ ROUGE-L 점수가 중간: 문장 구조 생성 능력을 개선할 필요가 있습니다.")
else:
    print("❌ ROUGE-L 점수가 낮음: 모델의 문장 구조 생성 능력이 부족합니다.")

print("\n" + "=" * 80)
print("🎯 개선 방향 제안")
print("=" * 80)

print("""
### 1. 데이터셋 개선
- ✅ 더 많은 학습 데이터 수집
- ✅ 데이터 품질 검증 및 정제
- ✅ 도메인별 균형 있는 데이터 분포 확보

### 2. 파인튜닝 최적화
- ✅ Learning Rate 조정
- ✅ Epoch 수 증가
- ✅ LoRA rank 파라미터 조정
- ✅ Batch size 최적화

### 3. 프롬프트 엔지니어링
- ✅ 더 명확한 instruction 설계
- ✅ Few-shot learning 활용
- ✅ 도메인 특화 프롬프트 템플릿 개발

### 4. 평가 방법 다각화
- ✅ LLM-as-a-Judge 평가 추가
- ✅ 전문가 평가 수행
- ✅ 실제 사용자 피드백 수집
- ✅ 한국어 특화 평가 지표 개발

### 5. 모델 아키텍처 개선
- ✅ 더 큰 베이스 모델 사용
- ✅ Adapter 방식 변경 (LoRA → QLoRA)
- ✅ 앙상블 기법 적용

### 6. lm-evaluation-harness 활용 확장
- ✅ 표준 벤치마크 평가 추가 (KMMLU, KoBEST 등)
- ✅ 다양한 태스크 평가 (분류, 요약, Q&A 등)
- ✅ 다른 모델과의 성능 비교
""")


In [ ]:
### 7.5 결과 저장

# 결과를 CSV 파일로 저장
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path("evaluation_results")
output_dir.mkdir(exist_ok=True)

# 질병 데이터셋 결과 저장
illnesses_results_df = pd.DataFrame(illnesses_results)
illnesses_output_file = output_dir / f"illnesses_results_{timestamp}.csv"
illnesses_results_df.to_csv(illnesses_output_file, index=False, encoding='utf-8-sig')

# 영양 데이터셋 결과 저장
nutrition_results_df = pd.DataFrame(nutrition_results)
nutrition_output_file = output_dir / f"nutrition_results_{timestamp}.csv"
nutrition_results_df.to_csv(nutrition_output_file, index=False, encoding='utf-8-sig')

print(f"✅ 평가 결과 저장 완료!")
print(f"  - 질병 데이터셋: {illnesses_output_file}")
print(f"  - 영양 데이터셋: {nutrition_output_file}")

# 평균 점수 저장
summary = {
    "timestamp": timestamp,
    "model_name": MODEL_NAME,
    "illnesses_samples": len(illnesses_results),
    "nutrition_samples": len(nutrition_results),
    **{f"illnesses_{k.lower().replace(' ', '_').replace('-', '')}": v for k, v in illnesses_metrics.items()},
    **{f"nutrition_{k.lower().replace(' ', '_').replace('-', '')}": v for k, v in nutrition_metrics.items()},
    **{f"overall_{k.lower().replace(' ', '_').replace('-', '')}": v for k, v in overall_metrics.items()}
}

summary_file = output_dir / f"summary_{timestamp}.json"
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"  - 요약 정보: {summary_file}")


## 8. 요약 및 추가 학습 자료

### 📚 학습 내용 요약

이 강의에서는 다음 내용을 다뤘습니다:

1. **lm-evaluation-harness 소개**
   - EleutherAI의 표준 평가 프레임워크
   - 200개 이상의 벤치마크 태스크 지원
   
2. **파인튜닝 모델 평가**
   - Hugging Face에서 모델 로드
   - 커스텀 데이터셋으로 평가
   - ROUGE 지표를 사용한 생성 품질 평가
   
3. **결과 분석**
   - 정량적 평가 (ROUGE 점수)
   - 정성적 평가 (샘플 응답 확인)
   - 시각화를 통한 성능 분석
   
4. **개선 방향 제안**
   - 데이터셋 개선
   - 파인튜닝 최적화
   - 프롬프트 엔지니어링
   - 평가 방법 다각화

### 🔗 추가 학습 자료

#### 공식 문서
- [lm-evaluation-harness GitHub](https://github.com/EleutherAI/lm-evaluation-harness)
- [Hugging Face Transformers](https://huggingface.co/docs/transformers)
- [ROUGE 메트릭 설명](https://en.wikipedia.org/wiki/ROUGE_(metric))

#### 관련 논문
- "Language Models are Few-Shot Learners" (GPT-3 논문)
- "ROUGE: A Package for Automatic Evaluation of Summaries"
- "LoRA: Low-Rank Adaptation of Large Language Models"

#### 한국어 평가 벤치마크
- KMMLU (Korean Massive Multitask Language Understanding)
- KoBEST (Korean Benchmark Suite of Evaluation Tasks)
- KorNLI (Korean Natural Language Inference)

### 💡 다음 단계

1. **더 많은 샘플로 평가**
   ```python
   # num_samples 파라미터를 None으로 설정하여 전체 평가
   illnesses_results = evaluate_dataset(illnesses_eval, "질병 데이터셋", num_samples=None)
   nutrition_results = evaluate_dataset(nutrition_eval, "영양 데이터셋", num_samples=None)
   ```

2. **표준 벤치마크 평가**
   ```bash
   lm_eval --model hf \
       --model_args pretrained=good593/EXAONE-3.5-2.4B-fine-tuning \
       --tasks hellaswag,arc_easy,arc_challenge \
       --device cuda:0 \
       --batch_size 8
   ```

3. **다른 평가 도구 사용**
   - DeepEval (LLM-as-a-Judge)
   - HELM (Holistic Evaluation)
   - Custom evaluation framework

### 🎯 핵심 포인트

✅ **객관적 평가**: ROUGE 점수는 모델 성능의 객관적 지표를 제공합니다.

✅ **정성 평가 필수**: 숫자만으로는 부족하며, 실제 답변 내용을 확인해야 합니다.

✅ **지속적 개선**: 평가 결과를 바탕으로 모델과 데이터를 지속적으로 개선합니다.

✅ **도메인 특화**: 의료/건강 도메인에 특화된 평가가 중요합니다.

---

### 🙏 감사합니다!

이 강의를 통해 lm-evaluation-harness를 사용한 파인튜닝 모델 평가 방법을 익히셨습니다.
질문이나 피드백이 있으시면 언제든지 연락 주세요!
